# Validate Silver regional flood exposure

## Purpose

Validate Table 14, `02-silver.silver_region_flood_exposure`, independently of the transformation. The validator recomputes source accounting, the expected region × level grid, boundary resolution, and the projected region areas. It checks all seven data-quality attributes with executable checks: Consistency, Accuracy, Completeness, Auditability, Validity, Uniqueness, and Timeliness.

Evidence is persisted to `04-validation.silver_dq_results` **before** the STOP gate runs, so failed blocking evidence stays reviewable. The validator never changes Table 14.

## Cost

One grouped scan parses the selected MGB snapshot once. That scan also tests each mapped usable row against a single union of the safe region boundaries, so no row is duplicated. Boundary work touches only the small region-level feature set. All checks come from one metrics row, and the STOP gate reads the persisted results.

## 1. Declare validation expectations and read upstream context

**What:** declare the approved area CRS, the square-kilometre plausibility range, and historical reference counts once. Then summarize the published lineage, the newest terminal loads, the latest Bronze, CONFIG, and PSGC validation evidence, and the current PSGC and mapping versions.

**Why:** the approved CRS is an independent expectation in this notebook. The transformation's own constant is not trusted to validate itself. The overlap tolerance is read from the published `transformation_rule_version`, so validation uses exactly the tolerance that produced the table.

**Reference counts are FLAG evidence only:** 63,684 MGB rows, 1,815 blank MGB geometries, and 232 duplicate boundary-lineage rows come from the accepted October 2026 Bronze evidence. They never filter or repair data.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE mgb_mapping_source_system STRING DEFAULT 'DENR MGB flood susceptibility';
DECLARE OR REPLACE VARIABLE expected_area_calculation_srid INT DEFAULT 6933;
DECLARE OR REPLACE VARIABLE source_geometry_srid INT DEFAULT 4326;
DECLARE OR REPLACE VARIABLE square_metres_per_sqkm DOUBLE DEFAULT 1000000;
-- square kilometres per square degree between about 4 and 21 degrees north is near 11,500 to 12,300
DECLARE OR REPLACE VARIABLE min_sqkm_per_square_degree DOUBLE DEFAULT 10000;
DECLARE OR REPLACE VARIABLE max_sqkm_per_square_degree DOUBLE DEFAULT 13000;
-- published areas are rounded to four decimals, so recalculation allows slightly more than rounding
DECLARE OR REPLACE VARIABLE area_recalculation_tolerance_sqkm DOUBLE DEFAULT 0.001;
DECLARE OR REPLACE VARIABLE degree_screen_min_longitude DOUBLE DEFAULT 110.0;
DECLARE OR REPLACE VARIABLE degree_screen_max_longitude DOUBLE DEFAULT 130.0;
DECLARE OR REPLACE VARIABLE degree_screen_min_latitude DOUBLE DEFAULT 0.0;
DECLARE OR REPLACE VARIABLE degree_screen_max_latitude DOUBLE DEFAULT 25.0;
DECLARE OR REPLACE VARIABLE reference_mgb_rows BIGINT DEFAULT 63684;
DECLARE OR REPLACE VARIABLE reference_blank_geometry_rows BIGINT DEFAULT 1815;
DECLARE OR REPLACE VARIABLE reference_boundary_lineage_duplicates BIGINT DEFAULT 232;
DECLARE OR REPLACE VARIABLE validation_rule_version STRING
    DEFAULT 'silver-region-flood-exposure-validation-v1';

CREATE OR REPLACE TEMPORARY VIEW flood_exposure_validation_context AS
WITH published AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT run_id) AS run_count,
        MAX(run_id) AS run_id,
        MAX(mgb_source_snapshot_id) AS mgb_source_snapshot_id,
        MAX(boundary_source_snapshot_id) AS boundary_source_snapshot_id,
        MAX(psgc_source_snapshot_id) AS psgc_source_snapshot_id,
        MAX(psgc_run_id) AS psgc_run_id,
        MAX(susceptibility_mapping_version) AS susceptibility_mapping_version,
        MAX(transformation_rule_version) AS transformation_rule_version,
        MAX(TRY_CAST(REGEXP_EXTRACT(
            transformation_rule_version, 'overlap-tolerance-sqkm=([0-9.Ee+-]+)', 1
        ) AS DOUBLE)) AS overlap_tolerance_sqkm
    FROM `02-silver`.silver_region_flood_exposure
),
latest_load AS (
    SELECT * EXCEPT (terminal_rank)
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
        FROM `01-bronze`.load_log
        WHERE table_name IN ('flood_susceptibility', 'boundaries')
          AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
    )
    WHERE terminal_rank = 1
),
load_summary AS (
    SELECT
        COUNT_IF(
            status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
            AND snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> ''
            AND rows_loaded > 0
        ) AS safe_latest_loads,
        MAX(CASE WHEN table_name = 'flood_susceptibility' THEN snapshot_id END) AS latest_mgb_snapshot_id,
        MAX(CASE WHEN table_name = 'flood_susceptibility' THEN rows_loaded END) AS latest_mgb_rows_loaded,
        MAX(CASE WHEN table_name = 'boundaries' THEN snapshot_id END) AS latest_boundary_snapshot_id,
        CONCAT_WS('; ', SORT_ARRAY(COLLECT_LIST(CONCAT(table_name, '=', status)))) AS latest_load_statuses
    FROM latest_load
),
bronze_validation AS (
    SELECT
        COUNT(DISTINCT table_name) AS validated_source_tables,
        COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) AS bronze_blocking_failures
    FROM (
        SELECT dq.*, DENSE_RANK() OVER (
            PARTITION BY dq.table_name ORDER BY dq.run_ts DESC, dq.run_id DESC
        ) AS validation_rank
        FROM `04-validation`.dq_results AS dq
        INNER JOIN latest_load AS load
            ON dq.table_name = load.table_name
           AND dq.snapshot_id = load.snapshot_id
    )
    WHERE validation_rank = 1
),
config_validation AS (
    SELECT
        COUNT(*) AS config_checks,
        COUNT_IF(action = 'stop' AND status = 'FAIL') AS config_blocking_failures
    FROM (
        SELECT *, DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
        FROM `04-validation`.silver_config_dq_results
        WHERE layer = 'silver_config'
    )
    WHERE validation_rank = 1
),
current_psgc AS (
    SELECT
        COUNT(*) AS official_region_count,
        MAX(source_snapshot_id) AS current_psgc_snapshot_id,
        MAX(run_id) AS current_psgc_run_id
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'REGION'
),
psgc_validation AS (
    SELECT
        COUNT(*) AS psgc_checks,
        COUNT_IF(dq.action = 'stop' AND dq.status = 'FAIL') AS psgc_blocking_failures
    FROM `04-validation`.silver_dq_results AS dq
    CROSS JOIN current_psgc
    WHERE dq.table_name = 'silver_psgc_place'
      AND dq.pipeline_run_id = current_psgc.current_psgc_run_id
),
current_mapping AS (
    SELECT
        COUNT(DISTINCT mapping_version) AS active_mapping_versions,
        MAX(mapping_version) AS current_mapping_version
    FROM `02-silver`.config_mgb_susceptibility_mapping
    WHERE source_system = mgb_mapping_source_system
      AND approval_status = 'APPROVED'
      AND is_active = TRUE
)
SELECT *
FROM published
CROSS JOIN load_summary
CROSS JOIN bronze_validation
CROSS JOIN config_validation
CROSS JOIN current_psgc
CROSS JOIN psgc_validation
CROSS JOIN current_mapping;

SELECT * FROM flood_exposure_validation_context;

## 2. Recompute region boundary evidence

**What:** reparse only the region-level features of the published boundary snapshot and attach the published status, feature count, and region area for each code.

**Why:** these features prove four things. The published boundary status matches the source. The published denominator equals a fresh equal-area recalculation. The denominator is square kilometres, not square degrees. The safe-region union drives the MGB intersection check.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flood_exposure_validation_published_regions AS
SELECT
    psgc_region_code,
    MAX(psgc_region_name) AS psgc_region_name,
    MAX(region_boundary_status) AS published_boundary_status,
    MAX(boundary_feature_count) AS published_feature_count,
    MAX(region_area_sqkm) AS published_region_area_sqkm,
    MAX(region_cross_level_overlap_sqkm) AS published_cross_level_overlap_sqkm,
    MAX(level_area_additivity_status) AS published_additivity_status,
    COUNT(DISTINCT region_area_sqkm) AS region_area_values,
    COUNT(DISTINCT region_cross_level_overlap_sqkm) AS cross_level_overlap_values,
    COUNT(DISTINCT level_area_additivity_status) AS additivity_values,
    COUNT(DISTINCT region_boundary_status) AS boundary_status_values
FROM `02-silver`.silver_region_flood_exposure
GROUP BY psgc_region_code;

CREATE OR REPLACE TEMPORARY VIEW flood_exposure_validation_boundaries AS
SELECT
    boundary.psgc_code AS boundary_psgc_code,
    current_region.psgc_code AS current_region_code,
    published.published_boundary_status,
    published.published_region_area_sqkm,
    TRY_TO_GEOMETRY(REGEXP_REPLACE(
        REGEXP_REPLACE(
            boundary.geometry_json,
            '\"(\\[[^\"]*\\])\"',
            CONCAT(CHAR(36), '1')
        ),
        '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"',
        CONCAT(CHAR(36), '1')
    )) AS boundary_geometry
FROM `01-bronze`.boundaries AS boundary
CROSS JOIN flood_exposure_validation_context AS context
LEFT JOIN `02-silver`.silver_psgc_place AS current_region
    ON boundary.psgc_code = current_region.psgc_code
   AND current_region.place_type = 'REGION'
LEFT JOIN flood_exposure_validation_published_regions AS published
    ON boundary.psgc_code = published.psgc_region_code
WHERE LOWER(TRIM(boundary.administrative_level)) = 'region'
  AND boundary._source_snapshot_id = context.boundary_source_snapshot_id;

## 3. Recompute MGB source accounting in one scan

**What:** classify every row of the published MGB snapshot with the published mapping version and the same parsing rules as the transformation. Group the rows by raw code, mapping status, and geometry status.

**Why:** these buckets must reconcile to the full selected row count and the audited `rows_loaded`. Blank and `No rating` stay in their own raw-code buckets. They are never assigned a standard level. Each mapped usable row is tested once against the union of safe region boundaries. That gives contributing and non-intersecting row counts without inventing a polygon key or duplicating rows across regions.

In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW flood_exposure_validation_mgb_buckets AS

WITH approved AS (
    SELECT
        UPPER(TRIM(mapping.raw_susceptibility_code)) AS raw_susceptibility_code,
        mapping.standardized_level
    FROM `buildabida-capstone`.`02-silver`.config_mgb_susceptibility_mapping AS mapping
    CROSS JOIN flood_exposure_validation_context AS context
    WHERE mapping.source_system = mgb_mapping_source_system
      AND mapping.approval_status = 'APPROVED'
      AND mapping.is_active = TRUE
      AND mapping.mapping_version = context.susceptibility_mapping_version
),

safe_region_union AS (
    SELECT
        ST_UNION_AGG(boundary_geometry) AS safe_region_geometry
    FROM flood_exposure_validation_boundaries
    WHERE published_boundary_status = 'VALID_REGION_BOUNDARY'
),

source_rows AS (
    SELECT
        NULLIF(
            UPPER(TRIM(mgb.flood_susceptibility_code)),
            ''
        ) AS raw_susceptibility_code,
        NULLIF(
            TRIM(mgb.geometry_json),
            ''
        ) AS geometry_text
    FROM `buildabida-capstone`.`01-bronze`.flood_susceptibility AS mgb
    CROSS JOIN flood_exposure_validation_context AS context
    WHERE mgb._source_snapshot_id = context.mgb_source_snapshot_id
),

-- try the geometry exactly as published first
direct_parse AS (
    SELECT
        esri_text.*,
        -- an Esri JSON row is never GeoJSON, so the GeoJSON readers skip it
        CASE
            WHEN esri_text.esri_rings_text IS NULL
            THEN TRY_TO_GEOMETRY(esri_text.geometry_text)
        END AS published_geometry
    FROM (
        SELECT
            source_rows.*,
            -- Esri JSON keeps its coordinates as text under the "rings" key
            NULLIF(REGEXP_EXTRACT(source_rows.geometry_text, '(?s)"rings"\\s*:\\s*(\\[.*\\])', 1), '') AS esri_rings_text
        FROM source_rows
    ) AS esri_text
),
esri_ring_source AS (
    SELECT
        direct_parse.*,
        -- first reading, built as text: every ring is its own polygon.
        -- Text avoids FROM_JSON arrays, which run out of memory on the largest MGB polygons.
        CASE
            WHEN esri_rings_text IS NULL THEN NULL
            WHEN esri_rings_text RLIKE '\\]\\s*\\]\\s*,\\s*\\[\\s*\\['
            THEN TRY_TO_GEOMETRY(CONCAT(
                '{"type":"MultiPolygon","coordinates":[',
                REGEXP_REPLACE(esri_rings_text, '\\]\\s*\\]\\s*,\\s*\\[\\s*\\[', ']]],[[['),
                ']}'
            ))
            ELSE TRY_TO_GEOMETRY(CONCAT('{"type":"Polygon","coordinates":', esri_rings_text, '}'))
        END AS esri_ring_geometry
    FROM direct_parse
),
esri_ring_checked AS (
    SELECT
        esri_ring_source.*,
        esri_ring_geometry IS NOT NULL AND ST_ISVALID(esri_ring_geometry) AS esri_ring_geometry_is_valid
    FROM esri_ring_source
),
esri_ring_arrays AS (
    SELECT
        esri_ring_checked.*,
        -- ring arrays are read only when the first reading is invalid, to find the holes
        CASE
            WHEN esri_rings_text IS NOT NULL AND NOT esri_ring_geometry_is_valid
            THEN FROM_JSON(esri_rings_text, 'ARRAY<ARRAY<ARRAY<DOUBLE>>>')
        END AS esri_rings
    FROM esri_ring_checked
),
esri_ring_parts AS (
    SELECT
        esri_ring_arrays.*,
        -- shoelace signed area per ring: Esri outer rings run clockwise (negative), holes counterclockwise
        TRANSFORM(
            esri_rings,
            ring -> AGGREGATE(
                SEQUENCE(1, SIZE(ring) - 1),
                CAST(0 AS DOUBLE),
                (area, i) -> area + ring[i - 1][0] * ring[i][1] - ring[i][0] * ring[i - 1][1]
            ) / 2
        ) AS esri_ring_signed_areas
    FROM esri_ring_arrays
),
esri_json_parsed AS (
    SELECT
        esri_ring_parts.*,
        CASE
            WHEN esri_ring_parts.esri_rings_text IS NOT NULL
            THEN ST_TRANSFORM(
                ST_SETSRID(
                    CASE
                        -- a valid first reading is kept as it is
                        WHEN esri_ring_geometry_is_valid
                        THEN esri_ring_geometry
                        -- otherwise read the counterclockwise rings as holes in the clockwise rings,
                        -- as the Esri polygon format defines them; this follows the format, it is not a repair
                        WHEN SIZE(FILTER(esri_ring_signed_areas, area -> area < 0)) > 0
                            AND SIZE(FILTER(esri_ring_signed_areas, area -> area > 0)) > 0
                        THEN ST_DIFFERENCE(
                            TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                                'type', 'MultiPolygon',
                                'coordinates', TRANSFORM(
                                    FILTER(
                                        ZIP_WITH(esri_rings, esri_ring_signed_areas, (ring, area) -> NAMED_STRUCT('ring', ring, 'area', area)),
                                        part -> part.area < 0
                                    ),
                                    part -> ARRAY(part.ring)
                                )
                            ))),
                            TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                                'type', 'MultiPolygon',
                                'coordinates', TRANSFORM(
                                    FILTER(
                                        ZIP_WITH(esri_rings, esri_ring_signed_areas, (ring, area) -> NAMED_STRUCT('ring', ring, 'area', area)),
                                        part -> part.area > 0
                                    ),
                                    part -> ARRAY(part.ring)
                                )
                            )))
                        )
                        ELSE esri_ring_geometry
                    END,
                    3857
                ),
                4326
            )
            ELSE NULL
        END AS esri_geometry
    FROM esri_ring_parts
),
normalized AS (
    SELECT
        esri_json_parsed.*,
        CASE
            WHEN esri_rings_text IS NULL
            THEN TRY_TO_GEOMETRY(REGEXP_REPLACE(
                REGEXP_REPLACE(geometry_text, '\"(\\[[^\"]*\\])\"', CONCAT(CHAR(36), '1')),
                '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"',
                CONCAT(CHAR(36), '1')
            ))
        END AS normalized_geometry
    FROM esri_json_parsed
),

-- resolve one final geometry using the same parser precedence as silver
parsed AS (
    SELECT
        normalized.*,
        COALESCE(
            published_geometry,
            normalized_geometry,
            esri_geometry
        ) AS mgb_geometry
    FROM normalized
),

classified AS (
    SELECT
        parsed.raw_susceptibility_code,
        parsed.mgb_geometry,

        parsed.published_geometry IS NOT NULL
            AS parsed_as_published,

        parsed.published_geometry IS NULL
            AND parsed.normalized_geometry IS NOT NULL
            AS parsed_after_normalization,

        parsed.published_geometry IS NULL
            AND parsed.normalized_geometry IS NULL
            AND parsed.esri_geometry IS NOT NULL
            AS parsed_as_esri_json,

        CASE
            WHEN approved.raw_susceptibility_code IS NOT NULL
                THEN 'MAPPED_APPROVED'
            ELSE 'UNMAPPED'
        END AS susceptibility_mapping_status,

        CASE
            WHEN parsed.geometry_text IS NULL
                THEN 'BLANK_GEOMETRY'

            WHEN parsed.mgb_geometry IS NULL
                THEN 'UNPARSEABLE_GEOMETRY'

            WHEN ST_ISEMPTY(parsed.mgb_geometry)
                THEN 'EMPTY_GEOMETRY'

            WHEN REGEXP_REPLACE(
                UPPER(ST_GEOMETRYTYPE(parsed.mgb_geometry)),
                '^ST_',
                ''
            ) NOT IN ('POLYGON', 'MULTIPOLYGON')
                THEN 'UNSUPPORTED_GEOMETRY_TYPE'

            WHEN NOT ST_ISVALID(parsed.mgb_geometry)
                THEN 'INVALID_GEOMETRY'

            ELSE 'USABLE_GEOMETRY'
        END AS source_geometry_status

    FROM parsed

    LEFT JOIN approved
        ON parsed.raw_susceptibility_code
            = approved.raw_susceptibility_code
),

tested AS (
    SELECT
        classified.*,

        CASE
            WHEN classified.susceptibility_mapping_status
                    = 'MAPPED_APPROVED'
             AND classified.source_geometry_status
                    = 'USABLE_GEOMETRY'
            THEN ST_INTERSECTS(
                safe_region_union.safe_region_geometry,
                classified.mgb_geometry
            )
        END AS intersects_safe_region,

        CASE
            WHEN classified.source_geometry_status
                    = 'USABLE_GEOMETRY'
            THEN ST_SRID(classified.mgb_geometry)
                <> source_geometry_srid
        END AS has_unexpected_srid,

        CASE
            WHEN classified.source_geometry_status
                    = 'USABLE_GEOMETRY'
            THEN
                ST_XMIN(classified.mgb_geometry)
                    < degree_screen_min_longitude
                OR ST_XMAX(classified.mgb_geometry)
                    > degree_screen_max_longitude
                OR ST_YMIN(classified.mgb_geometry)
                    < degree_screen_min_latitude
                OR ST_YMAX(classified.mgb_geometry)
                    > degree_screen_max_latitude
        END AS is_outside_degree_screen

    FROM classified
    CROSS JOIN safe_region_union
)

SELECT
    COALESCE(
        raw_susceptibility_code,
        '<BLANK>'
    ) AS raw_susceptibility_code,

    susceptibility_mapping_status,
    source_geometry_status,

    COUNT(*) AS source_rows,

    COUNT_IF(
        parsed_as_published
    ) AS parsed_as_published_rows,

    COUNT_IF(
        parsed_after_normalization
    ) AS parsed_after_normalization_rows,

    COUNT_IF(
        parsed_as_esri_json
    ) AS esri_json_converted_rows,

    COUNT_IF(
        intersects_safe_region
    ) AS intersecting_rows,

    COUNT_IF(
        NOT intersects_safe_region
    ) AS non_intersecting_rows,

    COUNT_IF(
        has_unexpected_srid
    ) AS unexpected_srid_rows,

    COUNT_IF(
        is_outside_degree_screen
    ) AS outside_degree_screen_rows

FROM tested

GROUP BY
    1,
    2,
    3;

## 4. Calculate one grouped metric row

**What:** combine upstream, source-accounting, grid, uniqueness, boundary, area, CRS, overlap, and lineage metrics into one row.

**Why:** every check has an explicit denominator. Expected grid size is derived as official regions × approved levels from the current inputs. Region and level counts are never hardcoded.

**Territory evidence:** province-level boundary codes are compared with the current PSGC hierarchy. A province code that sits under a resolved region in the older boundary snapshot is flagged when the current PSGC places it in another region or no longer lists it. A current PSGC province of the region that the boundary snapshot does not hold is also flagged, so a province that joined the region after the boundary was published is visible. A difference must lead to `BOUNDARY_TERRITORY_MISMATCH` unless a missing, ambiguous, or invalid boundary status already applies. This makes a boundary that predates a regional reorganization visible as a FLAG. Nothing is corrected.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flood_exposure_validation_metrics AS
WITH context AS (
    SELECT * FROM flood_exposure_validation_context
),
source_metrics AS (
    SELECT
        COALESCE(SUM(source_rows), 0) AS mgb_rows,
        COALESCE(SUM(CASE WHEN susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status = 'USABLE_GEOMETRY' THEN source_rows END), 0) AS mapped_usable_rows,
        COALESCE(SUM(CASE WHEN susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status <> 'USABLE_GEOMETRY' THEN source_rows END), 0) AS mapped_unusable_rows,
        COALESCE(SUM(CASE WHEN susceptibility_mapping_status = 'UNMAPPED'
            AND source_geometry_status = 'USABLE_GEOMETRY' THEN source_rows END), 0) AS unmapped_usable_rows,
        COALESCE(SUM(CASE WHEN susceptibility_mapping_status = 'UNMAPPED'
            AND source_geometry_status <> 'USABLE_GEOMETRY' THEN source_rows END), 0) AS unmapped_unusable_rows,
        COALESCE(SUM(CASE WHEN raw_susceptibility_code = '<BLANK>' THEN source_rows END), 0) AS blank_code_rows,
        COALESCE(SUM(CASE WHEN raw_susceptibility_code = 'NO RATING' THEN source_rows END), 0) AS no_rating_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status = 'USABLE_GEOMETRY' THEN source_rows END), 0) AS usable_geometry_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status = 'BLANK_GEOMETRY' THEN source_rows END), 0) AS blank_geometry_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status = 'UNPARSEABLE_GEOMETRY' THEN source_rows END), 0) AS unparseable_geometry_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status IN ('EMPTY_GEOMETRY', 'UNSUPPORTED_GEOMETRY_TYPE')
            THEN source_rows END), 0) AS empty_or_unsupported_geometry_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status = 'INVALID_GEOMETRY' THEN source_rows END), 0) AS invalid_geometry_rows,
        COALESCE(SUM(parsed_as_published_rows), 0) AS parsed_as_published_rows,
        COALESCE(SUM(parsed_after_normalization_rows), 0) AS parsed_after_normalization_rows,
        COALESCE(SUM(esri_json_converted_rows), 0) AS esri_json_converted_rows,
        COALESCE(SUM(intersecting_rows), 0) AS contributing_mgb_rows,
        COALESCE(SUM(non_intersecting_rows), 0) AS non_intersecting_mgb_rows,
        COALESCE(SUM(unexpected_srid_rows), 0) AS unexpected_srid_rows,
        COALESCE(SUM(outside_degree_screen_rows), 0) AS outside_degree_screen_rows,
        COALESCE(CONCAT_WS('; ', SORT_ARRAY(COLLECT_LIST(
            CONCAT(raw_susceptibility_code, '/', source_geometry_status, '=', source_rows)
        ))), '') AS source_bucket_details
    FROM flood_exposure_validation_mgb_buckets
),
approved_levels AS (
    SELECT
        mapping.standardized_level AS flood_susceptibility_level,
        mapping.severity_rank,
        CONCAT_WS(',', SORT_ARRAY(COLLECT_SET(UPPER(TRIM(mapping.raw_susceptibility_code)))))
            AS source_susceptibility_code
    FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
    CROSS JOIN context
    WHERE mapping.source_system = mgb_mapping_source_system
      AND mapping.approval_status = 'APPROVED'
      AND mapping.is_active = TRUE
      AND mapping.mapping_version = context.susceptibility_mapping_version
    GROUP BY mapping.standardized_level, mapping.severity_rank
),
mapping_metrics AS (
    SELECT
        COUNT(*) AS approved_mapping_rows,
        COUNT(*) - COUNT(DISTINCT UPPER(TRIM(mapping.raw_susceptibility_code))) AS duplicate_approved_codes,
        COUNT(DISTINCT mapping.standardized_level) AS approved_level_count,
        COUNT(DISTINCT mapping.severity_rank) AS approved_rank_count,
        COUNT(DISTINCT NAMED_STRUCT('level', mapping.standardized_level, 'rank', mapping.severity_rank))
            AS approved_level_rank_pairs,
        COUNT_IF(UPPER(TRIM(mapping.raw_susceptibility_code)) IN ('', 'NO RATING')) AS approved_unrated_codes
    FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
    CROSS JOIN context
    WHERE mapping.source_system = mgb_mapping_source_system
      AND mapping.approval_status = 'APPROVED'
      AND mapping.is_active = TRUE
      AND mapping.mapping_version = context.susceptibility_mapping_version
),
expected_grid AS (
    SELECT region.psgc_code AS psgc_region_code, level.*
    FROM `02-silver`.silver_psgc_place AS region
    CROSS JOIN approved_levels AS level
    WHERE region.place_type = 'REGION'
),
grid_metrics AS (
    SELECT
        (SELECT COUNT(*) FROM expected_grid) AS expected_grid_rows,
        (SELECT COUNT(*) FROM expected_grid AS expected
         LEFT ANTI JOIN `02-silver`.silver_region_flood_exposure AS published
             ON expected.psgc_region_code = published.psgc_region_code
            AND expected.flood_susceptibility_level = published.flood_susceptibility_level
        ) AS missing_grid_rows,
        (SELECT COUNT(*) FROM `02-silver`.silver_region_flood_exposure AS published
         LEFT ANTI JOIN expected_grid AS expected
             ON expected.psgc_region_code = published.psgc_region_code
            AND expected.flood_susceptibility_level = published.flood_susceptibility_level
        ) AS extra_grid_rows,
        (SELECT COUNT(*) FROM `02-silver`.silver_region_flood_exposure AS published
         INNER JOIN approved_levels AS level
             ON published.flood_susceptibility_level = level.flood_susceptibility_level
         WHERE published.severity_rank <> level.severity_rank
            OR published.source_susceptibility_code <> level.source_susceptibility_code
        ) AS level_contract_mismatches
),
output_metrics AS (
    SELECT
        COUNT(*) - COUNT(DISTINCT published.region_flood_exposure_key) AS duplicate_keys,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'region', published.psgc_region_code,
            'level', published.flood_susceptibility_level,
            'run', published.run_id
        )) AS duplicate_grain_rows,
        COUNT_IF(published.region_flood_exposure_key <> SHA2(CONCAT_WS(
            '|', published.psgc_region_code, published.flood_susceptibility_level, published.run_id
        ), 256)) AS invalid_keys,
        COUNT_IF(published.run_id <> SHA2(CONCAT_WS(
            '||',
            published.mgb_source_snapshot_id, published.mgb_source_ingest_run_id,
            COALESCE(published.mgb_source_version, 'NO_PUBLISHER_VERSION'),
            published.susceptibility_mapping_version, published.mgb_mapping_source_version,
            published.boundary_source_snapshot_id, published.boundary_source_ingest_run_id,
            COALESCE(published.boundary_version, 'NO_BOUNDARY_VERSION'),
            published.psgc_source_snapshot_id, published.psgc_run_id,
            COALESCE(published.psgc_version, 'NO_PUBLISHER_VERSION'),
            published.area_calculation_crs, published.area_rule_version,
            published.transformation_rule_version
        ), 256)) AS invalid_run_ids,
        COUNT_IF(published.spatial_match_status = 'CALCULATED') AS calculated_rows,
        COUNT_IF(published.spatial_match_status = 'NO_MAPPED_EXPOSURE') AS zero_exposure_rows,
        COUNT_IF(published.spatial_match_status = 'NO_SAFE_REGION_BOUNDARY') AS no_data_rows,
        COUNT_IF(
            published.susceptibility_mapping_status IS DISTINCT FROM 'MAPPED_APPROVED'
            OR published.region_boundary_status IS NULL
            OR published.region_boundary_status NOT IN (
                'VALID_REGION_BOUNDARY', 'NO_SAFE_REGION_BOUNDARY',
                'AMBIGUOUS_REGION_BOUNDARY', 'INVALID_REGION_GEOMETRY',
                'BOUNDARY_TERRITORY_MISMATCH'
            )
            OR published.spatial_match_status IS NULL
            OR published.spatial_match_status NOT IN (
                'CALCULATED', 'NO_MAPPED_EXPOSURE', 'NO_SAFE_REGION_BOUNDARY'
            )
            OR published.boundary_version_status IS NULL
            OR published.boundary_version_status NOT IN ('VERSION_ALIGNED', 'VERSION_MISMATCH_VISIBLE')
            OR published.source_geometry_status IS NULL
            OR published.source_geometry_status NOT IN (
                'VALID_SOURCE_GEOMETRY_USED', 'NO_INTERSECTING_VALID_GEOMETRY', 'NOT_EVALUATED'
            )
            OR published.level_area_additivity_status IS NULL
            OR published.level_area_additivity_status NOT IN (
                'ADDITIVE_WITHIN_TOLERANCE', 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP', 'NOT_EVALUATED'
            )
        ) AS invalid_status_values,
        COUNT_IF(
            (published.region_boundary_status = 'VALID_REGION_BOUNDARY'
                AND published.spatial_match_status NOT IN ('CALCULATED', 'NO_MAPPED_EXPOSURE'))
            OR (published.region_boundary_status <> 'VALID_REGION_BOUNDARY'
                AND published.spatial_match_status <> 'NO_SAFE_REGION_BOUNDARY')
            OR (published.spatial_match_status = 'CALCULATED'
                AND published.source_geometry_status <> 'VALID_SOURCE_GEOMETRY_USED')
            OR (published.spatial_match_status = 'NO_MAPPED_EXPOSURE'
                AND published.source_geometry_status <> 'NO_INTERSECTING_VALID_GEOMETRY')
            OR (published.spatial_match_status = 'NO_SAFE_REGION_BOUNDARY'
                AND (published.source_geometry_status <> 'NOT_EVALUATED'
                     OR published.exception_reason IS NULL))
            OR (published.spatial_match_status <> 'NO_SAFE_REGION_BOUNDARY'
                AND published.exception_reason IS NOT NULL)
        ) AS inconsistent_status_rows,
        COUNT_IF(published.spatial_match_status = 'NO_SAFE_REGION_BOUNDARY' AND (
            published.region_area_sqkm IS NOT NULL OR published.susceptible_area_sqkm IS NOT NULL
            OR published.share_of_region_area_pct IS NOT NULL OR published.source_polygon_count IS NOT NULL
            OR published.intersection_fragment_count IS NOT NULL
            OR published.within_level_overlap_sqkm IS NOT NULL
            OR published.region_cross_level_overlap_sqkm IS NOT NULL
            OR published.level_area_additivity_status <> 'NOT_EVALUATED'
        )) AS no_data_published_as_value,
        COUNT_IF(published.spatial_match_status = 'NO_MAPPED_EXPOSURE' AND (
            published.susceptible_area_sqkm IS DISTINCT FROM 0
            OR published.share_of_region_area_pct IS DISTINCT FROM 0
            OR published.source_polygon_count IS DISTINCT FROM 0
            OR published.intersection_fragment_count IS DISTINCT FROM 0
            OR published.within_level_overlap_sqkm IS DISTINCT FROM 0
        )) AS invalid_zero_exposure_rows,
        COUNT_IF(published.spatial_match_status IN ('CALCULATED', 'NO_MAPPED_EXPOSURE')
            AND (published.region_area_sqkm IS NULL OR published.region_area_sqkm <= 0)) AS nonpositive_region_area_rows,
        COUNT_IF(published.spatial_match_status IN ('CALCULATED', 'NO_MAPPED_EXPOSURE')
            AND (published.susceptible_area_sqkm IS NULL OR published.susceptible_area_sqkm < 0)) AS negative_susceptible_area_rows,
        COUNT_IF(published.spatial_match_status IN ('CALCULATED', 'NO_MAPPED_EXPOSURE')
            AND published.susceptible_area_sqkm > published.region_area_sqkm + context.overlap_tolerance_sqkm)
            AS area_outside_region_rows,
        COUNT_IF(published.spatial_match_status IN ('CALCULATED', 'NO_MAPPED_EXPOSURE')
            AND (published.share_of_region_area_pct IS NULL
                 OR published.share_of_region_area_pct < 0
                 OR published.share_of_region_area_pct > 100)) AS share_out_of_range_rows,
        COUNT_IF(published.spatial_match_status = 'CALCULATED' AND (
            published.source_polygon_count IS NULL OR published.source_polygon_count <= 0
            OR published.source_polygon_count <> published.intersection_fragment_count
        )) AS invalid_polygon_count_rows,
        COUNT_IF(published.within_level_overlap_sqkm < -context.overlap_tolerance_sqkm)
            AS negative_within_level_overlap_rows,
        COUNT_IF(published.within_level_overlap_sqkm > context.overlap_tolerance_sqkm)
            AS within_level_overlap_rows,
        COALESCE(SUM(published.within_level_overlap_sqkm), 0) AS within_level_overlap_sqkm,
        COUNT_IF(published.region_boundary_status = 'VALID_REGION_BOUNDARY' AND (
            published.region_cross_level_overlap_sqkm IS NULL
            OR published.region_cross_level_overlap_sqkm < -context.overlap_tolerance_sqkm
            OR (published.level_area_additivity_status = 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP'
                AND published.region_cross_level_overlap_sqkm <= context.overlap_tolerance_sqkm)
            OR (published.level_area_additivity_status = 'ADDITIVE_WITHIN_TOLERANCE'
                AND published.region_cross_level_overlap_sqkm > context.overlap_tolerance_sqkm)
        )) AS invalid_additivity_rows,
        COUNT_IF(
            published.area_calculation_crs IS NULL
            OR published.area_calculation_crs <> CONCAT('EPSG:', expected_area_calculation_srid)
            OR published.area_calculation_crs = CONCAT('EPSG:', source_geometry_srid)
            OR published.area_rule_version IS NULL
            OR INSTR(published.area_rule_version, published.area_calculation_crs) = 0
        ) AS invalid_crs_rows,
        COUNT_IF(
            published.region_flood_exposure_key IS NULL OR published.psgc_region_code IS NULL
            OR published.psgc_region_name IS NULL OR published.flood_susceptibility_level IS NULL
            OR published.severity_rank IS NULL OR published.source_susceptibility_code IS NULL
            OR published.mgb_source_snapshot_id IS NULL OR published.mgb_source_ingest_run_id IS NULL
            OR published.mgb_source_load_ts IS NULL
            OR published.mgb_mapping_source_version IS NULL
            OR published.susceptibility_mapping_version IS NULL OR published.mapping_rule IS NULL
            OR published.boundary_source_snapshot_id IS NULL
            OR published.boundary_source_ingest_run_id IS NULL
            OR published.psgc_source_snapshot_id IS NULL OR published.psgc_source_ingest_run_id IS NULL
            OR published.psgc_run_id IS NULL OR published.psgc_source_load_ts IS NULL
            OR published.area_calculation_crs IS NULL OR published.area_rule_version IS NULL
            OR published.transformation_rule_version IS NULL
            OR published.run_id IS NULL OR published.source_load_ts IS NULL
            OR (published.spatial_match_status IN ('CALCULATED', 'NO_MAPPED_EXPOSURE') AND (
                published.boundary_source_system IS NULL
                OR published.boundary_source_load_ts IS NULL
                OR published.boundary_feature_count IS DISTINCT FROM 1
            ))
        ) AS missing_mandatory_lineage_rows,
        COUNT_IF(published.mgb_source_version IS NULL OR TRIM(published.mgb_source_version) = '')
            AS missing_mgb_publisher_version_rows,
        COUNT_IF(
            published.boundary_version IS NULL OR TRIM(published.boundary_version) = ''
            OR published.boundary_psgc_version IS NULL OR TRIM(published.boundary_psgc_version) = ''
        ) AS missing_boundary_version_rows,
        COUNT_IF(published.boundary_version_status = 'VERSION_MISMATCH_VISIBLE') AS version_mismatch_rows,
        MAX(published.boundary_psgc_version) AS boundary_psgc_version,
        MAX(published.psgc_version) AS psgc_version,
        COALESCE(SUM(published.source_polygon_count), 0) AS regional_source_polygon_sum
    FROM `02-silver`.silver_region_flood_exposure AS published
    CROSS JOIN context
),
level_totals AS (
    SELECT COALESCE(CONCAT_WS('; ', SORT_ARRAY(COLLECT_LIST(CONCAT(
        LPAD(CAST(severity_rank AS STRING), 2, '0'), ' ', flood_susceptibility_level,
        ' sqkm=', CAST(level_area_sqkm AS STRING)
    )))), '') AS level_area_details
    FROM (
        SELECT severity_rank, flood_susceptibility_level, SUM(susceptible_area_sqkm) AS level_area_sqkm
        FROM `02-silver`.silver_region_flood_exposure
        GROUP BY severity_rank, flood_susceptibility_level
    )
),
region_metrics AS (
    SELECT
        COUNT(*) AS published_region_count,
        COUNT_IF(published_boundary_status = 'VALID_REGION_BOUNDARY') AS safe_region_count,
        COUNT_IF(published_boundary_status <> 'VALID_REGION_BOUNDARY') AS no_safe_region_count,
        COUNT_IF(published_boundary_status = 'AMBIGUOUS_REGION_BOUNDARY') AS ambiguous_region_count,
        COUNT_IF(published_boundary_status = 'INVALID_REGION_GEOMETRY') AS invalid_region_geometry_count,
        COUNT_IF(region_area_values > 1) AS regions_with_multiple_denominators,
        COUNT_IF(cross_level_overlap_values > 1 OR additivity_values > 1 OR boundary_status_values > 1)
            AS regions_with_inconsistent_region_values,
        COUNT_IF(published_additivity_status = 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP') AS non_additive_region_count,
        COALESCE(SUM(published_cross_level_overlap_sqkm), 0) AS cross_level_overlap_sqkm,
        COALESCE(CONCAT_WS('; ', SORT_ARRAY(COLLECT_LIST(CASE
            WHEN published_boundary_status <> 'VALID_REGION_BOUNDARY'
                THEN CONCAT(psgc_region_code, ' ', psgc_region_name, ' (', published_boundary_status, ')')
        END))), '') AS no_safe_region_details,
        COALESCE(CONCAT_WS('; ', SORT_ARRAY(COLLECT_LIST(CASE
            WHEN published_additivity_status = 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP'
                THEN CONCAT(psgc_region_code, ' overlap sqkm=', CAST(published_cross_level_overlap_sqkm AS STRING))
        END))), '') AS non_additive_region_details
    FROM flood_exposure_validation_published_regions
),
boundary_metrics AS (
    SELECT
        COUNT(*) AS region_boundary_candidates,
        COUNT_IF(current_region_code IS NULL) AS unresolved_boundary_rows,
        COUNT_IF(published_boundary_status = 'VALID_REGION_BOUNDARY') AS recomputed_safe_boundaries,
        COUNT_IF(CASE
            WHEN published_boundary_status = 'VALID_REGION_BOUNDARY'
                THEN ABS(
                    ST_AREA(ST_TRANSFORM(boundary_geometry, expected_area_calculation_srid))
                    / square_metres_per_sqkm - published_region_area_sqkm
                ) > area_recalculation_tolerance_sqkm
        END) AS region_area_recalculation_mismatches,
        COUNT_IF(CASE
            -- square degrees are read only as a reference to prove the published unit is square kilometres
            WHEN published_boundary_status = 'VALID_REGION_BOUNDARY'
                THEN published_region_area_sqkm / NULLIF(ST_AREA(boundary_geometry), 0)
                    NOT BETWEEN min_sqkm_per_square_degree AND max_sqkm_per_square_degree
        END) AS square_degree_unit_violations
    FROM flood_exposure_validation_boundaries
),
boundary_status_metrics AS (
    SELECT COUNT_IF(
        published.published_feature_count <> COALESCE(candidate.feature_count, 0)
        OR (published.published_boundary_status = 'NO_SAFE_REGION_BOUNDARY'
            AND COALESCE(candidate.feature_count, 0) <> 0)
        OR (published.published_boundary_status = 'AMBIGUOUS_REGION_BOUNDARY'
            AND COALESCE(candidate.feature_count, 0) <= 1)
        OR (published.published_boundary_status IN ('VALID_REGION_BOUNDARY', 'INVALID_REGION_GEOMETRY', 'BOUNDARY_TERRITORY_MISMATCH')
            AND COALESCE(candidate.feature_count, 0) <> 1)
    ) AS boundary_status_mismatches
    FROM flood_exposure_validation_published_regions AS published
    LEFT JOIN (
        SELECT boundary_psgc_code, COUNT(*) AS feature_count
        FROM flood_exposure_validation_boundaries
        GROUP BY boundary_psgc_code
    ) AS candidate
        ON published.psgc_region_code = candidate.boundary_psgc_code
),
full_boundary_metrics AS (
    SELECT
        COUNT(*) AS boundary_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'source_file', boundary.source_file, 'source_feature_id', boundary.source_feature_id
        )) AS duplicate_boundary_lineage_rows
    FROM `01-bronze`.boundaries AS boundary
    CROSS JOIN context
    WHERE boundary._source_snapshot_id = context.boundary_source_snapshot_id
),
territory_boundary_province AS (
    SELECT DISTINCT boundary.psgc_code
    FROM `01-bronze`.boundaries AS boundary
    CROSS JOIN context
    WHERE boundary._source_snapshot_id = context.boundary_source_snapshot_id
      AND boundary.psgc_code RLIKE '^[0-9]{5}00000$'
      AND boundary.psgc_code NOT RLIKE '^[0-9]{2}00000000$'
),
-- a province the boundary places in the region that current PSGC moved or no longer lists
territory_left AS (
    SELECT
        region.psgc_region_code,
        COUNT(*) AS province_level_features_examined,
        COUNT_IF(
            current_place.psgc_code IS NULL
            OR current_place.region_code <> region.psgc_region_code
        ) AS left_count,
        COLLECT_SET(
            CASE
                WHEN current_place.psgc_code IS NULL
                    OR current_place.region_code <> region.psgc_region_code
                THEN CONCAT(
                    region.psgc_region_code,
                    ' contains ',
                    boundary_province.psgc_code,
                    ' now ',
                    COALESCE(current_place.region_code, 'unlisted')
                )
            END
        ) AS left_details
    FROM territory_boundary_province AS boundary_province
    INNER JOIN flood_exposure_validation_published_regions AS region
        ON SUBSTRING(boundary_province.psgc_code, 1, 2)
            = SUBSTRING(region.psgc_region_code, 1, 2)
    LEFT JOIN `02-silver`.silver_psgc_place AS current_place
        ON boundary_province.psgc_code = current_place.psgc_code
    GROUP BY region.psgc_region_code
),
-- a current PSGC province of the region that the boundary does not hold
territory_joined AS (
    SELECT
        region.psgc_region_code,
        COUNT(*) AS joined_count,
        COLLECT_SET(CONCAT(region.psgc_region_code, ' lacks ', current_place.psgc_code))
            AS joined_details
    FROM `02-silver`.silver_psgc_place AS current_place
    INNER JOIN flood_exposure_validation_published_regions AS region
        ON current_place.region_code = region.psgc_region_code
    LEFT ANTI JOIN territory_boundary_province AS boundary_province
        ON current_place.psgc_code = boundary_province.psgc_code
    WHERE current_place.psgc_code = current_place.province_code
    GROUP BY region.psgc_region_code
),
territory_detail AS (
    SELECT
        COALESCE(territory_left.psgc_region_code, territory_joined.psgc_region_code)
            AS psgc_region_code,
        COALESCE(territory_left.province_level_features_examined, 0)
            AS province_level_features_examined,
        COALESCE(territory_left.left_count, 0) + COALESCE(territory_joined.joined_count, 0)
            AS territory_difference_count,
        CONCAT_WS(
            '; ',
            SORT_ARRAY(CONCAT(
                COALESCE(territory_left.left_details, CAST(ARRAY() AS ARRAY<STRING>)),
                COALESCE(territory_joined.joined_details, CAST(ARRAY() AS ARRAY<STRING>))
            ))
        ) AS territory_difference_details
    FROM territory_left
    FULL OUTER JOIN territory_joined
        ON territory_left.psgc_region_code = territory_joined.psgc_region_code
),
territory_metrics AS (
    SELECT
        COALESCE(
            SUM(territory.province_level_features_examined),
            0
        ) AS province_level_features_examined,
        COUNT_IF(
            COALESCE(territory.territory_difference_count, 0) > 0
        ) AS regions_with_territory_difference,
        COUNT_IF(
            -- missing, ambiguous, and invalid statuses take precedence over a territory mismatch
            (
                COALESCE(territory.territory_difference_count, 0) > 0
                AND published.published_boundary_status = 'VALID_REGION_BOUNDARY'
            )
            OR (
                COALESCE(territory.territory_difference_count, 0) = 0
                AND published.published_boundary_status
                    = 'BOUNDARY_TERRITORY_MISMATCH'
            )
        ) AS territory_status_mismatches,
        COALESCE(
            CONCAT_WS(
                '; ',
                SORT_ARRAY(
                    COLLECT_SET(
                        CASE
                            WHEN COALESCE(
                                territory.territory_difference_count,
                                0
                            ) > 0
                            THEN territory.territory_difference_details
                        END
                    )
                )
            ),
            ''
        ) AS territory_difference_details
    FROM flood_exposure_validation_published_regions AS published
    LEFT JOIN territory_detail AS territory
        ON published.psgc_region_code = territory.psgc_region_code
)
SELECT *
FROM context
CROSS JOIN source_metrics
CROSS JOIN mapping_metrics
CROSS JOIN grid_metrics
CROSS JOIN output_metrics
CROSS JOIN level_totals
CROSS JOIN region_metrics
CROSS JOIN boundary_metrics
CROSS JOIN boundary_status_metrics
CROSS JOIN full_boundary_metrics
CROSS JOIN territory_metrics;

## 5. Express STOP and FLAG checks across all seven attributes

**STOP** protects upstream safety and timeliness, source-row accounting, mapping governance, the complete grid, grain and key uniqueness, and status consistency. It also covers zero-versus-no-data semantics, area validity, the equal-area CRS, clipping, dissolve, additivity, denominators, polygon counts, lineage, and deterministic identity.

**FLAG** retains real source limitations with denominators. These include unmapped and unrated MGB rows, unusable or invalid geometry, and polygons outside safe regions. They also cover missing or ambiguous region boundaries, version mismatch, territory differences, overlap findings, optional version gaps, and differences from historical references. A FLAG is not a failure.

A failed blocking check is `FAIL`. A nonzero review finding is `FLAG`. Zero findings are `PASS`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flood_exposure_validation_checks AS
WITH checks AS (
    -- one metrics row is unpivoted into check rows, so the source scan is not repeated per check
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Latest MGB and boundary loads are successful and identified',
            'failed_rows', CAST(2 - safe_latest_loads AS BIGINT), 'total_rows', CAST(2 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Latest terminal loads: ', latest_load_statuses)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Latest MGB and boundary Bronze validation has no blocking failure',
            'failed_rows', CAST(bronze_blocking_failures + (2 - validated_source_tables) AS BIGINT), 'total_rows', CAST(2 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Validated source tables=', validated_source_tables, '; blocking failures=', bronze_blocking_failures)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published MGB snapshot is the latest validated MGB snapshot',
            'failed_rows', CAST(CASE WHEN mgb_source_snapshot_id <=> latest_mgb_snapshot_id THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published=', COALESCE(mgb_source_snapshot_id, 'NULL'), '; latest=', COALESCE(latest_mgb_snapshot_id, 'NULL'))
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published boundary snapshot is the latest validated boundary snapshot',
            'failed_rows', CAST(CASE WHEN boundary_source_snapshot_id <=> latest_boundary_snapshot_id THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published=', COALESCE(boundary_source_snapshot_id, 'NULL'), '; latest=', COALESCE(latest_boundary_snapshot_id, 'NULL'))
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published PSGC lineage is the current Silver PSGC run',
            'failed_rows', CAST(CASE WHEN psgc_source_snapshot_id <=> current_psgc_snapshot_id AND psgc_run_id <=> current_psgc_run_id THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published PSGC snapshot=', COALESCE(psgc_source_snapshot_id, 'NULL'), '; current=', COALESCE(current_psgc_snapshot_id, 'NULL'))
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published mapping version is the single current approved version',
            'failed_rows', CAST(CASE WHEN active_mapping_versions = 1 AND susceptibility_mapping_version <=> current_mapping_version THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published=', COALESCE(susceptibility_mapping_version, 'NULL'), '; current=', COALESCE(current_mapping_version, 'NULL'), '; active approved versions=', active_mapping_versions)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'One deterministic run represents the table',
            'failed_rows', CAST(ABS(run_count - 1) AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Distinct run IDs=', run_count)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Runtime MGB counts are compared with historical Bronze references',
            'failed_rows', CAST(ABS(mgb_rows - reference_mgb_rows) + ABS(blank_geometry_rows - reference_blank_geometry_rows) AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Runtime rows=', mgb_rows, ' versus reference ', reference_mgb_rows, '; runtime blank geometry=', blank_geometry_rows, ' versus reference ', reference_blank_geometry_rows, '; references are evidence, not filters')
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Silver CONFIG validation is present and has no blocking failure',
            'failed_rows', CAST(config_blocking_failures + CASE WHEN config_checks = 0 THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(GREATEST(config_checks, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Latest CONFIG checks=', config_checks, '; blocking failures=', config_blocking_failures)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Selected PSGC validation is present and has no blocking failure',
            'failed_rows', CAST(psgc_blocking_failures + CASE WHEN psgc_checks = 0 THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(GREATEST(psgc_checks, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Selected PSGC checks=', psgc_checks, '; blocking failures=', psgc_blocking_failures)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Approved MGB mapping has one target per raw code and one-to-one levels and ranks',
            'failed_rows', CAST(duplicate_approved_codes + ABS(approved_level_count - approved_rank_count) + ABS(approved_level_count - approved_level_rank_pairs) + CASE WHEN approved_mapping_rows = 0 THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(GREATEST(approved_mapping_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Approved rows=', approved_mapping_rows, '; levels=', approved_level_count, '; ranks=', approved_rank_count, '; duplicate codes=', duplicate_approved_codes)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Spatial, boundary, and geometry statuses agree',
            'failed_rows', CAST(inconsistent_status_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with inconsistent status combinations=', inconsistent_status_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Published boundary statuses match the selected boundary snapshot',
            'failed_rows', CAST(boundary_status_mismatches AS BIGINT), 'total_rows', CAST(published_region_count AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Regions whose status or feature count disagrees with the boundary source=', boundary_status_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Each region uses one area denominator across levels',
            'failed_rows', CAST(regions_with_multiple_denominators AS BIGINT), 'total_rows', CAST(published_region_count AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Regions with more than one region_area_sqkm=', regions_with_multiple_denominators)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Region-level overlap, additivity, and boundary values are identical across levels',
            'failed_rows', CAST(regions_with_inconsistent_region_values AS BIGINT), 'total_rows', CAST(published_region_count AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Regions with inconsistent region-level values=', regions_with_inconsistent_region_values)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Cross-level additivity status matches measured overlap',
            'failed_rows', CAST(invalid_additivity_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Invalid additivity rows=', invalid_additivity_rows, '; tolerance sqkm=', COALESCE(CAST(overlap_tolerance_sqkm AS STRING), 'NOT_RECORDED'))
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Dissolve never increases area within a level',
            'failed_rows', CAST(negative_within_level_overlap_rows + CASE WHEN overlap_tolerance_sqkm IS NULL THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows where dissolved area exceeds summed fragment area beyond tolerance=', negative_within_level_overlap_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Source polygon and fragment counts follow their contract',
            'failed_rows', CAST(invalid_polygon_count_rows AS BIGINT), 'total_rows', CAST(calculated_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Calculated rows with nonpositive or unequal source and fragment counts=', invalid_polygon_count_rows, '; regional source_polygon_count sum=', regional_source_polygon_sum, ' and may exceed unique rows because border-crossing polygons count in each region')
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Boundary and PSGC version differences remain visible',
            'failed_rows', CAST(version_mismatch_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Boundary PSGC version=', COALESCE(boundary_psgc_version, 'NULL'), '; current PSGC version=', COALESCE(psgc_version, 'NULL'), '; mismatch rows=', version_mismatch_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Published levels, ranks, and codes come only from approved mapping',
            'failed_rows', CAST(level_contract_mismatches AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows whose rank or source code differs from the approved level=', level_contract_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Approval of blank or No rating codes remains visible',
            'failed_rows', CAST(approved_unrated_codes AS BIGINT), 'total_rows', CAST(GREATEST(approved_mapping_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Approved mapping rows for blank or No rating=', approved_unrated_codes, '; zero means these values stay unresolved')
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Published areas use the approved projected equal-area CRS',
            'failed_rows', CAST(invalid_crs_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Expected EPSG:', expected_area_calculation_srid, '; rows with missing, wrong, or square-degree CRS=', invalid_crs_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Published region areas are square kilometres, not square degrees',
            'failed_rows', CAST(square_degree_unit_violations AS BIGINT), 'total_rows', CAST(recomputed_safe_boundaries AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Safe regions whose sqkm-per-square-degree ratio is outside ', min_sqkm_per_square_degree, ' to ', max_sqkm_per_square_degree, '=', square_degree_unit_violations)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Published region areas match an independent equal-area recalculation',
            'failed_rows', CAST(region_area_recalculation_mismatches AS BIGINT), 'total_rows', CAST(recomputed_safe_boundaries AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Safe regions differing by more than ', area_recalculation_tolerance_sqkm, ' sqkm=', region_area_recalculation_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Exposure is clipped inside its region boundary',
            'failed_rows', CAST(area_outside_region_rows AS BIGINT), 'total_rows', CAST(calculated_rows + zero_exposure_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows where susceptible area exceeds region area beyond tolerance=', area_outside_region_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Usable MGB geometry uses the expected longitude-latitude SRID',
            'failed_rows', CAST(unexpected_srid_rows AS BIGINT), 'total_rows', CAST(usable_geometry_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Usable rows with an unexpected SRID=', unexpected_srid_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Usable MGB geometry outside the degree-unit screen remains visible',
            'failed_rows', CAST(outside_degree_screen_rows AS BIGINT), 'total_rows', CAST(usable_geometry_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Usable rows outside the coarse longitude-latitude screen=', outside_degree_screen_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Territory-mismatched boundaries are never treated as safe geography',
            'failed_rows', CAST(territory_status_mismatches AS BIGINT), 'total_rows', CAST(published_region_count AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Regions whose published boundary status disagrees with current PSGC territory evidence=', territory_status_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Region boundary territory differences from current PSGC remain visible',
            'failed_rows', CAST(regions_with_territory_difference AS BIGINT), 'total_rows', CAST(safe_region_count AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Province-level boundary features examined=', province_level_features_examined, '; differences: ', territory_difference_details)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Within-level source overlap removed by dissolve remains visible',
            'failed_rows', CAST(within_level_overlap_rows AS BIGINT), 'total_rows', CAST(calculated_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Region-level rows with overlap removed=', within_level_overlap_rows, '; total removed sqkm=', CAST(within_level_overlap_sqkm AS STRING))
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Cross-level overlap between approved levels remains visible',
            'failed_rows', CAST(non_additive_region_count AS BIGINT), 'total_rows', CAST(safe_region_count AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Total cross-level overlap sqkm=', CAST(cross_level_overlap_sqkm AS STRING), '; do not sum level areas for: ', non_additive_region_details)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'MGB source rows reconcile to mapping and geometry buckets',
            'failed_rows', CAST(ABS(mgb_rows - mapped_usable_rows - mapped_unusable_rows - unmapped_usable_rows - unmapped_unusable_rows) + ABS(mgb_rows - COALESCE(latest_mgb_rows_loaded, 0)) AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Selected rows=', mgb_rows, '; audited rows=', COALESCE(latest_mgb_rows_loaded, 0), '; mapped usable=', mapped_usable_rows, '; mapped unusable=', mapped_unusable_rows, '; unmapped usable=', unmapped_usable_rows, '; unmapped unusable=', unmapped_unusable_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Every official region and approved level has exactly one output row',
            'failed_rows', CAST(missing_grid_rows + extra_grid_rows + ABS(output_rows - expected_grid_rows) AS BIGINT), 'total_rows', CAST(expected_grid_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Official regions=', official_region_count, ' x approved levels=', approved_level_count, ' = expected ', expected_grid_rows, '; actual=', output_rows, '; missing=', missing_grid_rows, '; extra=', extra_grid_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Unmapped or unrated MGB rows remain visible',
            'failed_rows', CAST(unmapped_usable_rows + unmapped_unusable_rows AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Blank code=', blank_code_rows, '; No rating=', no_rating_rows, '; other unmapped=', unmapped_usable_rows + unmapped_unusable_rows - blank_code_rows - no_rating_rows, '; buckets: ', source_bucket_details)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Mapped MGB rows without usable geometry remain visible',
            'failed_rows', CAST(mapped_unusable_rows AS BIGINT), 'total_rows', CAST(mapped_usable_rows + mapped_unusable_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Mapped rows excluded from area=', mapped_unusable_rows, '; mapped usable=', mapped_usable_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Mapped usable MGB rows without a safe region intersection remain visible',
            'failed_rows', CAST(non_intersecting_mgb_rows AS BIGINT), 'total_rows', CAST(mapped_usable_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Contributing rows=', contributing_mgb_rows, '; non-intersecting=', non_intersecting_mgb_rows, '; level totals: ', level_area_details)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Official regions without a safe boundary remain no data',
            'failed_rows', CAST(no_safe_region_count AS BIGINT), 'total_rows', CAST(published_region_count AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Safe regions=', safe_region_count, '; no-data regions: ', no_safe_region_details)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Mandatory snapshot, mapping, boundary, PSGC, CRS, rule, and run lineage is present',
            'failed_rows', CAST(missing_mandatory_lineage_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows missing mandatory lineage=', missing_mandatory_lineage_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Exposure keys reproduce from region, level, and run',
            'failed_rows', CAST(invalid_keys AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Invalid deterministic keys=', invalid_keys)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Run ID reproduces from published source, mapping, CRS, and rule lineage',
            'failed_rows', CAST(invalid_run_ids AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows whose run_id does not reproduce=', invalid_run_ids)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Optional MGB publisher version gaps remain visible',
            'failed_rows', CAST(missing_mgb_publisher_version_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Rows without a publisher version=', missing_mgb_publisher_version_rows, '; snapshot lineage remains mandatory')
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Optional boundary version gaps remain visible',
            'failed_rows', CAST(missing_boundary_version_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Rows without boundary or boundary PSGC version=', missing_boundary_version_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'MGB geometry parse paths are reported',
            'failed_rows', CAST(0 AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Parsed as published=', parsed_as_published_rows, '; parsed after quoted-number normalization=', parsed_after_normalization_rows, '; parsed from Esri rings=', esri_json_converted_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Status values use the controlled vocabulary',
            'failed_rows', CAST(invalid_status_values AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with uncontrolled status values=', invalid_status_values)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Missing geography is published as NULL, never zero',
            'failed_rows', CAST(no_data_published_as_value AS BIGINT), 'total_rows', CAST(GREATEST(no_data_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('No-data rows=', no_data_rows, '; no-data rows carrying a value=', no_data_published_as_value)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Safe regions without exposure are published as real zero',
            'failed_rows', CAST(invalid_zero_exposure_rows AS BIGINT), 'total_rows', CAST(GREATEST(zero_exposure_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Zero-exposure rows=', zero_exposure_rows, '; invalid zero rows=', invalid_zero_exposure_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Calculated region area is positive',
            'failed_rows', CAST(nonpositive_region_area_rows AS BIGINT), 'total_rows', CAST(calculated_rows + zero_exposure_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Safe rows with null or nonpositive region area=', nonpositive_region_area_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Susceptible area is non-negative',
            'failed_rows', CAST(negative_susceptible_area_rows AS BIGINT), 'total_rows', CAST(calculated_rows + zero_exposure_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Safe rows with null or negative susceptible area=', negative_susceptible_area_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Share of region area is between 0 and 100',
            'failed_rows', CAST(share_out_of_range_rows AS BIGINT), 'total_rows', CAST(calculated_rows + zero_exposure_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Safe rows with null or out-of-range share=', share_out_of_range_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Blank MGB geometry remains visible',
            'failed_rows', CAST(blank_geometry_rows AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Blank geometry=', blank_geometry_rows, '; historical reference=', reference_blank_geometry_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Unparseable MGB geometry remains visible',
            'failed_rows', CAST(unparseable_geometry_rows AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Unparseable geometry=', unparseable_geometry_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Empty or unsupported MGB geometry remains visible',
            'failed_rows', CAST(empty_or_unsupported_geometry_rows AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Empty or non-polygon geometry=', empty_or_unsupported_geometry_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Invalid MGB geometry is excluded without repair',
            'failed_rows', CAST(invalid_geometry_rows AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('OGC-invalid geometry excluded from area=', invalid_geometry_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Unresolved region boundary codes remain visible',
            'failed_rows', CAST(unresolved_boundary_rows AS BIGINT), 'total_rows', CAST(region_boundary_candidates AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Region-level boundary features without a current PSGC region=', unresolved_boundary_rows, '; candidates=', region_boundary_candidates)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Ambiguous or invalid region boundaries remain visible',
            'failed_rows', CAST(ambiguous_region_count + invalid_region_geometry_count AS BIGINT), 'total_rows', CAST(published_region_count AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Ambiguous regions=', ambiguous_region_count, '; invalid region geometry=', invalid_region_geometry_count)
        ),
        NAMED_STRUCT(
            'attribute', 'Uniqueness', 'check_name', 'Region flood exposure keys are unique',
            'failed_rows', CAST(duplicate_keys AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Duplicate keys=', duplicate_keys)
        ),
        NAMED_STRUCT(
            'attribute', 'Uniqueness', 'check_name', 'One row exists per region, level, and run',
            'failed_rows', CAST(duplicate_grain_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Duplicate region, level, and run rows=', duplicate_grain_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Uniqueness', 'check_name', 'Full boundary source lineage duplicates remain visible',
            'failed_rows', CAST(duplicate_boundary_lineage_rows AS BIGINT), 'total_rows', CAST(boundary_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Duplicate source-file and feature rows=', duplicate_boundary_lineage_rows, '; historical reference=', reference_boundary_lineage_duplicates, '; the boundary source is not deduplicated')
        )
    ))
    FROM flood_exposure_validation_metrics
),
context AS (
    SELECT
        run_id AS pipeline_run_id,
        mgb_source_snapshot_id AS source_snapshot_id,
        transformation_rule_version AS rule_version,
        SHA2(CONCAT_WS('|', run_id, validation_rule_version), 256) AS validation_run_id,
        CURRENT_TIMESTAMP() AS run_ts
    FROM flood_exposure_validation_context
)
SELECT
    context.validation_run_id, context.pipeline_run_id, 'silver' AS layer,
    'silver_region_flood_exposure' AS table_name, context.source_snapshot_id, context.rule_version,
    checks.attribute AS data_quality_attribute, checks.check_name AS data_quality_check,
    checks.failed_rows, checks.total_rows,
    CASE WHEN checks.total_rows > 0
        THEN ROUND(100.0 * checks.failed_rows / checks.total_rows, 4) END AS percentage,
    CASE
        WHEN checks.failed_rows = 0 THEN 'PASS'
        WHEN checks.action = 'stop' THEN 'FAIL'
        ELSE 'FLAG'
    END AS status,
    checks.action, checks.details, context.run_ts
FROM checks
CROSS JOIN context;

## 6. Persist evidence before enforcing the gate

**What:** `MERGE` the checks into the shared `04-validation.silver_dq_results` table, then enforce the STOP gate from the persisted rows.

**Why:** the deterministic validation run ID updates the same evidence on an exact rerun instead of appending duplicates. The gate reads stored evidence, so the expensive checks run once and failed STOP evidence stays reviewable.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.silver_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Reusable Silver transformation validation evidence';

MERGE INTO `04-validation`.silver_dq_results AS target
USING flood_exposure_validation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

DECLARE OR REPLACE VARIABLE flood_exposure_validation_run_id STRING;
SET VAR flood_exposure_validation_run_id = (
    SELECT SHA2(CONCAT_WS('|', MAX(run_id), validation_rule_version), 256)
    FROM `02-silver`.silver_region_flood_exposure
);

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking Silver regional flood-exposure checks failed or no evidence was persisted'
    )
)
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_region_flood_exposure'
  AND validation_run_id = flood_exposure_validation_run_id;

## Summary

Validation protects the exact Table 14 grain, which is one official region × approved level × selected snapshots × deterministic run. It also protects complete MGB source-row accounting and approved-only mapping. Other STOP coverage includes the dynamic region × level grid, deterministic keys and run identity, boundary-status reproduction, and zero-versus-no-data semantics. Area checks prove the equal-area CRS, square-kilometre units, clipping, dissolve, additivity, consistent denominators, and complete lineage.

Unmapped and unrated MGB rows, unusable geometry, and missing or older boundaries remain visible as FLAG findings with denominators. Overlap findings and differences from historical references are also FLAG findings. Evidence is stored before the gate. Requires Databricks execution. No runtime result is claimed by this notebook text.

In [0]:
%sql
SELECT data_quality_attribute, status, action, failed_rows, total_rows, percentage,
       data_quality_check, details
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_region_flood_exposure'
  AND validation_run_id = flood_exposure_validation_run_id
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END,
         data_quality_attribute, data_quality_check;